# Polars on GPU: TPC-H (PDS-H) 22 queries from S3

The 22 PDS-H queries from the cuDF benchmark (`cudf_polars/streaming/benchmarks/pdsh.py`). Each cell loads its tables, builds a lazy `frame`, then runs it on the CPU (plain `.collect()`) and on the GPU (`RayEngine`) and prints the time with `%time`.

Needs all 8 TPC-H tables under `s3://rapids-tpch/scale-1000/<table>/`: lineitem, orders, customer, part, partsupp, supplier, nation, region. Run each query twice: the first GPU run includes warm-up.

In [1]:
import os, boto3

creds = boto3.Session().get_credentials().get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"] = creds.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = creds.secret_key
if creds.token:  # instance roles use temporary credentials
    os.environ["AWS_SESSION_TOKEN"] = creds.token
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"  # use the region of the rapids-tpch bucket

In [2]:
# Check the instance can read the bucket (shell commands)
!aws sts get-caller-identity
!aws s3 ls s3://rapids-tpch/scale-1000/lineitem/ | head
!aws s3 ls s3://rapids-tpch/scale-1000/lineitem/ --summarize --human-readable | tail -3

{
    "UserId": "AROAYFLEYUUALDRTPAA45:allisond@nvidia.com",
    "Account": "561241433344",
    "Arn": "arn:aws:sts::561241433344:assumed-role/AWSReservedSSO_CS-Engineer-561241433344_aceb91bc76004a35/allisond@nvidia.com"
}
2025-09-16 18:26:28  485500272 part.0.parquet
2025-09-16 18:26:28  485507127 part.1.parquet
2025-09-16 18:26:31  485749414 part.10.parquet
2025-09-16 18:26:31  485824711 part.100.parquet
2025-09-16 18:26:34  485828489 part.101.parquet
2025-09-16 18:26:38  485692314 part.102.parquet
2025-09-16 18:26:39  485254846 part.103.parquet
2025-09-16 18:26:47  485699512 part.104.parquet
2025-09-16 18:26:50  485517671 part.105.parquet
2025-09-16 18:26:52  485574602 part.106.parquet

aws: [ERROR]: [Errno 32] Broken pipe
Exception ignored while flushing sys.stdout:
BrokenPipeError: [Errno 32] Broken pipe

Total Objects: 500
   Total Size: 226.1 GiB


In [3]:
import polars as pl
from datetime import date

# Same settings the cuDF PDS-H benchmark uses (set BEFORE starting the engine)
os.environ["KVIKIO_COMPAT_MODE"] = "on"
os.environ["RAPIDSMPF_NUM_STREAMING_THREADS"] = "8"

BASE = "s3://rapids-tpch/scale-1000"

def get_data(table):
    return pl.scan_parquet(f"{BASE}/{table}/*.parquet")

# Quick check: lineitem schema
get_data("lineitem").collect_schema()


Schema([('l_orderkey', Int64),
        ('l_partkey', Int64),
        ('l_suppkey', Int64),
        ('l_linenumber', Int64),
        ('l_quantity', Float64),
        ('l_extendedprice', Float64),
        ('l_discount', Float64),
        ('l_tax', Float64),
        ('l_returnflag', String),
        ('l_linestatus', String),
        ('l_shipdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_commitdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_receiptdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_shipinstruct', String),
        ('l_shipmode', String),
        ('l_comment', String)])

In [4]:
# Start the GPU engine once and reuse it
from cudf_polars.engine.ray import RayEngine   # needs: pip install ray
engine = RayEngine()


2026-10-27 22:27:30,602	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
2026-10-27 22:27:33,043	INFO worker.py:2024 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


In [5]:
# ~150 GB of lineitem: first 309 files (~485 MB each)
# ~100 GB of lineitem: first 206 files (~485 MB each)
N_FILES = 206
files = [f"s3://rapids-tpch/scale-1000/lineitem/part.{i}.parquet" for i in range(N_FILES)]
lineitem = pl.scan_parquet(files)   # add the same storage_options/args you used for `lineitem`

lineitem.collect_schema()


Schema([('l_orderkey', Int64),
        ('l_partkey', Int64),
        ('l_suppkey', Int64),
        ('l_linenumber', Int64),
        ('l_quantity', Float64),
        ('l_extendedprice', Float64),
        ('l_discount', Float64),
        ('l_tax', Float64),
        ('l_returnflag', String),
        ('l_linestatus', String),
        ('l_shipdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_commitdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_receiptdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_shipinstruct', String),
        ('l_shipmode', String),
        ('l_comment', String)])

In [6]:
# Query 1

var1 = date(1998, 9, 2)

frame = (
    lineitem.filter(pl.col("l_shipdate") <= var1)
    .group_by("l_returnflag", "l_linestatus")
    .agg(
        pl.sum("l_quantity").alias("sum_qty"),
        pl.sum("l_extendedprice").alias("sum_base_price"),
        (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount")))
        .sum()
        .alias("sum_disc_price"),
        (
            pl.col("l_extendedprice")
            * (1.0 - pl.col("l_discount"))
            * (1.0 + pl.col("l_tax"))
        )
        .sum()
        .alias("sum_charge"),
        pl.mean("l_quantity").alias("avg_qty"),
        pl.mean("l_extendedprice").alias("avg_price"),
        pl.mean("l_discount").alias("avg_disc"),
        pl.len().alias("count_order"),
    )
    .sort("l_returnflag", "l_linestatus")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


CPU times: user 3min 53s, sys: 46.2 s, total: 4min 39s
Wall time: 19.4 s
CPU times: user 155 ms, sys: 937 ms, total: 1.09 s
Wall time: 15.2 s


l_returnflag,l_linestatus,sum_qty,sum_base_price,sum_disc_price,sum_charge,avg_qty,avg_price,avg_disc,count_order
str,str,f64,f64,f64,f64,f64,f64,f64,u64
"""A""","""F""",1.5556e10,2.3325e13,2.2159e13,2.3046e13,25.499913,38236.936451,0.050001,610023831
"""N""","""F""",4.05917363e8,6.0861e11,5.7818e11,6.0131e11,25.500016,38233.525613,0.050012,15918318
"""N""","""O""",3.0633e10,4.5934e13,4.3638e13,4.5383e13,25.50015,38237.308913,0.049999,1201295132
"""R""","""F""",1.5555e10,2.3325e13,2.2159e13,2.3045e13,25.500434,38237.922345,0.05,610003319


In [7]:
# Query 2
nation = get_data("nation")
part = get_data("part")
partsupp = get_data("partsupp")
region = get_data("region")
supplier = get_data("supplier")

var1 = 15
var2 = "BRASS"
var3 = "EUROPE"

q1 = (
    part.join(partsupp, left_on="p_partkey", right_on="ps_partkey")
    .join(supplier, left_on="ps_suppkey", right_on="s_suppkey")
    .join(nation, left_on="s_nationkey", right_on="n_nationkey")
    .join(region, left_on="n_regionkey", right_on="r_regionkey")
    .filter(pl.col("p_size") == var1)
    .filter(pl.col("p_type").str.ends_with(var2))
    .filter(pl.col("r_name") == var3)
)

frame = (
    q1.group_by("p_partkey")
    .agg(pl.min("ps_supplycost"))
    .join(q1, on=["p_partkey", "ps_supplycost"])
    .select(
        "s_acctbal",
        "s_name",
        "n_name",
        "p_partkey",
        "p_mfgr",
        "s_address",
        "s_phone",
        "s_comment",
    )
    .sort(
        by=["s_acctbal", "n_name", "s_name", "p_partkey"],
        descending=[True, False, False, False],
    )
    .head(100)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


CPU times: user 44.3 s, sys: 11.6 s, total: 55.8 s
Wall time: 4.47 s
CPU times: user 82.7 ms, sys: 483 ms, total: 566 ms
Wall time: 11.6 s


s_acctbal,s_name,n_name,p_partkey,p_mfgr,s_address,s_phone,s_comment
f64,str,str,i64,str,str,str,str
9999.98,"""Supplier#009603132""","""FRANCE""",124603107,"""Manufacturer#5""","""sWwhImcmui18""","""16-772-274-7879""","""se idly across the furiously r…"
9999.98,"""Supplier#009603132""","""FRANCE""",192103074,"""Manufacturer#3""","""sWwhImcmui18""","""16-772-274-7879""","""se idly across the furiously r…"
9999.98,"""Supplier#005781756""","""RUSSIA""",90781737,"""Manufacturer#1""","""mWkaVs2HYIMuMSyOm3z MNm6M""","""32-318-428-3941""","""ounts haggle slyly about the q…"
9999.97,"""Supplier#009095529""","""GERMANY""",6595528,"""Manufacturer#2""","""5CCKmqY5bOys0aBIK3""","""17-888-873-6604""","""tions. patterns nag busily bef…"
9999.95,"""Supplier#009454986""","""UNITED KINGDOM""",44454977,"""Manufacturer#1""","""yMUWlr1Y0Ba ,o4w0Hs,5fQD64M6Pc…","""33-670-600-8336""","""s sleep carefully. carefully s…"
…,…,…,…,…,…,…,…
9997.58,"""Supplier#009824039""","""GERMANY""",102324008,"""Manufacturer#1""",""",ahnSGkru,9Chpfu5VvR""","""17-203-917-5644""","""symptotes use fluffily expr"""
9997.58,"""Supplier#000991966""","""RUSSIA""",143491923,"""Manufacturer#4""","""IqRrBGCrKrQ2eZ""","""32-573-261-5414""","""pinto beans thrash furiously c…"
9997.57,"""Supplier#001712265""","""ROMANIA""",199212245,"""Manufacturer#5""","""cBo901jgqwwCJI""","""29-108-652-9788""","""ackages cajole carefully. blit…"


In [8]:
# Query 3
customer = get_data("customer")
lineitem = get_data("lineitem")
orders = get_data("orders")

var1 = "BUILDING"
var2 = date(1995, 3, 15)

frame = (
    customer.filter(pl.col("c_mktsegment") == var1)
    .join(orders, left_on="c_custkey", right_on="o_custkey")
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .filter(pl.col("o_orderdate") < var2)
    .filter(pl.col("l_shipdate") > var2)
    .with_columns(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).alias("revenue")
    )
    .group_by("o_orderkey", "o_orderdate", "o_shippriority")
    .agg(pl.sum("revenue"))
    .select(
        pl.col("o_orderkey").alias("l_orderkey"),
        "revenue",
        "o_orderdate",
        "o_shippriority",
    )
    .sort(by=["revenue", "o_orderdate"], descending=[True, False])
    .head(10)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


CPU times: user 12min 2s, sys: 3min 13s, total: 15min 15s
Wall time: 41 s
CPU times: user 553 ms, sys: 10.3 s, total: 10.9 s
Wall time: 55 s


l_orderkey,revenue,o_orderdate,o_shippriority
i64,f64,datetime[ms],i32
18869634,512508.6578,1995-01-10 00:00:00,0
3947421511,507889.0464,1995-03-14 00:00:00,0
1319897249,503401.9508,1995-03-05 00:00:00,0
2036965252,495852.8692,1995-03-03 00:00:00,0
1980912577,493605.4659,1995-02-14 00:00:00,0
4803840546,492521.943,1995-02-18 00:00:00,0
3407391428,491379.5186,1995-03-09 00:00:00,0
5289035781,488004.5812,1995-03-11 00:00:00,0
5530172133,487671.6623,1995-02-07 00:00:00,0


In [ ]:
# Query 4
lineitem = get_data("lineitem")
orders = get_data("orders")

var1 = date(1993, 7, 1)
var2 = date(1993, 10, 1)

frame = (
    # SQL exists translates to semi join in Polars API
    orders.join(
        (lineitem.filter(pl.col("l_commitdate") < pl.col("l_receiptdate"))),
        left_on="o_orderkey",
        right_on="l_orderkey",
        how="semi",
    )
    .filter(pl.col("o_orderdate").is_between(var1, var2, closed="left"))
    .group_by("o_orderpriority")
    .agg(pl.len().alias("order_count"))
    .sort("o_orderpriority")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


CPU times: user 9min 33s, sys: 1min 58s, total: 11min 32s
Wall time: 28.8 s


In [ ]:
# Query 5
customer = get_data("customer")
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")
region = get_data("region")
supplier = get_data("supplier")

var1 = "ASIA"
var2 = date(1994, 1, 1)
var3 = date(1995, 1, 1)

frame = (
    region.join(nation, left_on="r_regionkey", right_on="n_regionkey")
    .join(customer, left_on="n_nationkey", right_on="c_nationkey")
    .join(orders, left_on="c_custkey", right_on="o_custkey")
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .join(
        supplier,
        left_on=["l_suppkey", "n_nationkey"],
        right_on=["s_suppkey", "s_nationkey"],
    )
    .filter(pl.col("r_name") == var1)
    .filter(pl.col("o_orderdate").is_between(var2, var3, closed="left"))
    .with_columns(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).alias("revenue")
    )
    .group_by("n_name")
    .agg(pl.sum("revenue"))
    .sort(by="revenue", descending=True)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 6
lineitem = get_data("lineitem")

var1 = date(1994, 1, 1)
var2 = date(1995, 1, 1)
var3 = 0.05
var4 = 0.07
var5 = 24

frame = (
    lineitem.filter(pl.col("l_shipdate").is_between(var1, var2, closed="left"))
    .filter(pl.col("l_discount").is_between(var3, var4))
    .filter(pl.col("l_quantity") < var5)
    .with_columns(
        (pl.col("l_extendedprice") * pl.col("l_discount")).alias("revenue")
    )
    .select(pl.sum("revenue"))
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 7
customer = get_data("customer")
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")
supplier = get_data("supplier")

var1 = "FRANCE"
var2 = "GERMANY"
var3 = date(1995, 1, 1)
var4 = date(1996, 12, 31)

n1 = nation.filter(pl.col("n_name") == var1)
n2 = nation.filter(pl.col("n_name") == var2)

q1 = (
    customer.join(n1, left_on="c_nationkey", right_on="n_nationkey")
    .join(orders, left_on="c_custkey", right_on="o_custkey")
    .rename({"n_name": "cust_nation"})
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .join(supplier, left_on="l_suppkey", right_on="s_suppkey")
    .join(n2, left_on="s_nationkey", right_on="n_nationkey")
    .rename({"n_name": "supp_nation"})
)

q2 = (
    customer.join(n2, left_on="c_nationkey", right_on="n_nationkey")
    .join(orders, left_on="c_custkey", right_on="o_custkey")
    .rename({"n_name": "cust_nation"})
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .join(supplier, left_on="l_suppkey", right_on="s_suppkey")
    .join(n1, left_on="s_nationkey", right_on="n_nationkey")
    .rename({"n_name": "supp_nation"})
)

frame = (
    pl.concat([q1, q2])
    .filter(pl.col("l_shipdate").is_between(var3, var4))
    .with_columns(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).alias("volume"),
        pl.col("l_shipdate").dt.year().alias("l_year"),
    )
    .group_by("supp_nation", "cust_nation", "l_year")
    .agg(pl.sum("volume").alias("revenue"))
    .sort(by=["supp_nation", "cust_nation", "l_year"])
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 8
customer = get_data("customer")
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")
part = get_data("part")
region = get_data("region")
supplier = get_data("supplier")

var1 = "BRAZIL"
var2 = "AMERICA"
var3 = "ECONOMY ANODIZED STEEL"
var4 = date(1995, 1, 1)
var5 = date(1996, 12, 31)

n1 = nation.select("n_nationkey", "n_regionkey")
n2 = nation.select("n_nationkey", "n_name")

frame = (
    part.join(lineitem, left_on="p_partkey", right_on="l_partkey")
    .join(supplier, left_on="l_suppkey", right_on="s_suppkey")
    .join(orders, left_on="l_orderkey", right_on="o_orderkey")
    .join(customer, left_on="o_custkey", right_on="c_custkey")
    .join(n1, left_on="c_nationkey", right_on="n_nationkey")
    .join(region, left_on="n_regionkey", right_on="r_regionkey")
    .filter(pl.col("r_name") == var2)
    .join(n2, left_on="s_nationkey", right_on="n_nationkey")
    .filter(pl.col("o_orderdate").is_between(var4, var5))
    .filter(pl.col("p_type") == var3)
    .select(
        pl.col("o_orderdate").dt.year().alias("o_year"),
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).alias("volume"),
        pl.col("n_name").alias("nation"),
    )
    .with_columns(
        pl.when(pl.col("nation") == var1)
        .then(pl.col("volume"))
        .otherwise(0)
        .alias("_tmp")
    )
    .group_by("o_year")
    .agg((pl.sum("_tmp") / pl.sum("volume")).alias("mkt_share"))
    .sort("o_year")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 9
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")
part = get_data("part")
partsupp = get_data("partsupp")
supplier = get_data("supplier")

frame = (
    part.join(partsupp, left_on="p_partkey", right_on="ps_partkey")
    .join(supplier, left_on="ps_suppkey", right_on="s_suppkey")
    .join(
        lineitem,
        left_on=["p_partkey", "ps_suppkey"],
        right_on=["l_partkey", "l_suppkey"],
    )
    .join(orders, left_on="l_orderkey", right_on="o_orderkey")
    .join(nation, left_on="s_nationkey", right_on="n_nationkey")
    .filter(pl.col("p_name").str.contains("green"))
    .select(
        pl.col("n_name").alias("nation"),
        pl.col("o_orderdate").dt.year().alias("o_year"),
        (
            pl.col("l_extendedprice") * (1 - pl.col("l_discount"))
            - pl.col("ps_supplycost") * pl.col("l_quantity")
        ).alias("amount"),
    )
    .group_by("nation", "o_year")
    .agg(pl.sum("amount").round(2).alias("sum_profit"))
    .sort(by=["nation", "o_year"], descending=[False, True])
)

%time cpu = frame.collect()
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 10
customer = get_data("customer")
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")

var1 = date(1993, 10, 1)
var2 = date(1994, 1, 1)

frame = (
    customer.join(orders, left_on="c_custkey", right_on="o_custkey")
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .join(nation, left_on="c_nationkey", right_on="n_nationkey")
    .filter(pl.col("o_orderdate").is_between(var1, var2, closed="left"))
    .filter(pl.col("l_returnflag") == "R")
    .group_by(
        "c_custkey",
        "c_name",
        "c_acctbal",
        "c_phone",
        "n_name",
        "c_address",
        "c_comment",
    )
    .agg(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount")))
        .sum()
        .round(2)
        .alias("revenue")
    )
    .select(
        "c_custkey",
        "c_name",
        "revenue",
        "c_acctbal",
        "n_name",
        "c_address",
        "c_phone",
        "c_comment",
    )
    .sort(by="revenue", descending=True)
    .head(20)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 11
nation = get_data("nation")
partsupp = get_data("partsupp")
supplier = get_data("supplier")

var1 = "GERMANY"
var2 = float(f"{0.0001 / 1000:.12f}")

q1 = (
    partsupp.join(supplier, left_on="ps_suppkey", right_on="s_suppkey")
    .join(nation, left_on="s_nationkey", right_on="n_nationkey")
    .filter(pl.col("n_name") == var1)
)
q2 = q1.select(
    (pl.col("ps_supplycost") * pl.col("ps_availqty"))
    .sum()
    .round(2)
    .alias("tmp")
    * var2
)

frame = (
    q1.group_by("ps_partkey")
    .agg(
        (pl.col("ps_supplycost") * pl.col("ps_availqty"))
        .sum()
        .round(2)
        .alias("value")
    )
    .join(q2, how="cross")
    .filter(pl.col("value") > pl.col("tmp"))
    .select("ps_partkey", "value")
    .sort("value", descending=True)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 12
lineitem = get_data("lineitem")
orders = get_data("orders")

var1 = "MAIL"
var2 = "SHIP"
var3 = date(1994, 1, 1)
var4 = date(1995, 1, 1)

frame = (
    orders.join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .filter(pl.col("l_shipmode").is_in([var1, var2]))
    .filter(pl.col("l_commitdate") < pl.col("l_receiptdate"))
    .filter(pl.col("l_shipdate") < pl.col("l_commitdate"))
    .filter(pl.col("l_receiptdate").is_between(var3, var4, closed="left"))
    .with_columns(
        pl.when(pl.col("o_orderpriority").is_in(["1-URGENT", "2-HIGH"]))
        .then(1)
        .otherwise(0)
        .alias("high_line_count"),
        pl.when(pl.col("o_orderpriority").is_in(["1-URGENT", "2-HIGH"]).not_())
        .then(1)
        .otherwise(0)
        .alias("low_line_count"),
    )
    .group_by("l_shipmode")
    .agg(pl.col("high_line_count").sum(), pl.col("low_line_count").sum())
    .sort("l_shipmode")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 13
customer = get_data("customer")
orders = get_data("orders")

var1 = "special"
var2 = "requests"

orders = orders.filter(
    pl.col("o_comment").str.contains(f"{var1}.*{var2}").not_()
)

frame = (
    customer.join(orders, left_on="c_custkey", right_on="o_custkey", how="left")
    .group_by("c_custkey")
    .agg(pl.col("o_orderkey").count().alias("c_count"))
    .group_by("c_count")
    .len()
    .select(pl.col("c_count"), pl.col("len").alias("custdist"))
    .sort(by=["custdist", "c_count"], descending=[True, True])
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 14
lineitem = get_data("lineitem")
part = get_data("part")

var1 = date(1995, 9, 1)
var2 = date(1995, 10, 1)

frame = (
    lineitem.join(part, left_on="l_partkey", right_on="p_partkey")
    .filter(pl.col("l_shipdate").is_between(var1, var2, closed="left"))
    .select(
        (
            100.00
            * pl.when(pl.col("p_type").str.contains("PROMO*"))
            .then(pl.col("l_extendedprice") * (1 - pl.col("l_discount")))
            .otherwise(0)
            .sum()
            / (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum()
        )
        .round(2)
        .alias("promo_revenue")
    )
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 15
lineitem = get_data("lineitem")
supplier = get_data("supplier")

var1 = date(1996, 1, 1)
var2 = date(1996, 4, 1)

revenue = (
    lineitem.filter(pl.col("l_shipdate").is_between(var1, var2, closed="left"))
    .group_by("l_suppkey")
    .agg(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount")))
        .sum()
        .alias("total_revenue")
    )
    .select(pl.col("l_suppkey").alias("supplier_no"), pl.col("total_revenue"))
)

frame = (
    supplier.join(revenue, left_on="s_suppkey", right_on="supplier_no")
    .filter(pl.col("total_revenue") == pl.col("total_revenue").max())
    .with_columns(pl.col("total_revenue").round(2))
    .select("s_suppkey", "s_name", "s_address", "s_phone", "total_revenue")
    .sort("s_suppkey")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 16
part = get_data("part")
partsupp = get_data("partsupp")
supplier = get_data("supplier")

var1 = "Brand#45"

supplier = supplier.filter(
    pl.col("s_comment").str.contains(".*Customer.*Complaints.*")
).select(pl.col("s_suppkey"), pl.col("s_suppkey").alias("ps_suppkey"))

frame = (
    part.join(partsupp, left_on="p_partkey", right_on="ps_partkey")
    .filter(pl.col("p_brand") != var1)
    .filter(pl.col("p_type").str.contains("MEDIUM POLISHED*").not_())
    .filter(pl.col("p_size").is_in([49, 14, 23, 45, 19, 3, 36, 9]))
    .join(supplier, left_on="ps_suppkey", right_on="s_suppkey", how="left")
    .filter(pl.col("ps_suppkey_right").is_null())
    .group_by("p_brand", "p_type", "p_size")
    .agg(pl.col("ps_suppkey").n_unique().alias("supplier_cnt"))
    .sort(
        by=["supplier_cnt", "p_brand", "p_type", "p_size"],
        descending=[True, False, False, False],
    )
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 17
lineitem = get_data("lineitem")
part = get_data("part")

var1 = "Brand#23"
var2 = "MED BOX"

q1 = (
    part.filter(pl.col("p_brand") == var1)
    .filter(pl.col("p_container") == var2)
    .join(lineitem, how="inner", left_on="p_partkey", right_on="l_partkey")
)

frame = (
    q1.group_by("p_partkey")
    .agg((0.2 * pl.col("l_quantity").mean()).alias("avg_quantity"))
    .select(pl.col("p_partkey").alias("key"), pl.col("avg_quantity"))
    .join(q1, left_on="key", right_on="p_partkey")
    .filter(pl.col("l_quantity") < pl.col("avg_quantity"))
    .select(
        (pl.col("l_extendedprice").sum() / 7.0).round(2).alias("avg_yearly")
    )
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 18
customer = get_data("customer")
lineitem = get_data("lineitem")
orders = get_data("orders")

var1 = 300

q1 = (
    lineitem.group_by("l_orderkey")
    .agg(pl.col("l_quantity").sum().alias("sum_quantity"))
    .filter(pl.col("sum_quantity") > var1)
)

frame = (
    orders.join(q1, left_on="o_orderkey", right_on="l_orderkey", how="semi")
    .join(lineitem, left_on="o_orderkey", right_on="l_orderkey")
    .join(customer, left_on="o_custkey", right_on="c_custkey")
    .group_by(
        "c_name", "o_custkey", "o_orderkey", "o_orderdate", "o_totalprice"
    )
    .agg(pl.col("l_quantity").sum().alias("sum(l_quantity)"))
    .select(
        pl.col("c_name"),
        pl.col("o_custkey").alias("c_custkey"),
        pl.col("o_orderkey"),
        pl.col("o_orderdate"),
        pl.col("o_totalprice"),
        pl.col("sum(l_quantity)"),
    )
    .sort(by=["o_totalprice", "o_orderdate"], descending=[True, False])
    .head(100)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 19
lineitem = get_data("lineitem")
part = get_data("part")

frame = (
    part.join(lineitem, left_on="p_partkey", right_on="l_partkey")
    .filter(pl.col("l_shipmode").is_in(["AIR", "AIR REG"]))
    .filter(pl.col("l_shipinstruct") == "DELIVER IN PERSON")
    .filter(
        (
            (pl.col("p_brand") == "Brand#12")
            & pl.col("p_container").is_in(
                ["SM CASE", "SM BOX", "SM PACK", "SM PKG"]
            )
            & (pl.col("l_quantity").is_between(1, 11))
            & (pl.col("p_size").is_between(1, 5))
        )
        | (
            (pl.col("p_brand") == "Brand#23")
            & pl.col("p_container").is_in(
                ["MED BAG", "MED BOX", "MED PKG", "MED PACK"]
            )
            & (pl.col("l_quantity").is_between(10, 20))
            & (pl.col("p_size").is_between(1, 10))
        )
        | (
            (pl.col("p_brand") == "Brand#34")
            & pl.col("p_container").is_in(
                ["LG CASE", "LG BOX", "LG PACK", "LG PKG"]
            )
            & (pl.col("l_quantity").is_between(20, 30))
            & (pl.col("p_size").is_between(1, 15))
        )
    )
    .select(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount")))
        .sum()
        .round(2)
        .alias("revenue")
    )
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 20
lineitem = get_data("lineitem")
nation = get_data("nation")
part = get_data("part")
partsupp = get_data("partsupp")
supplier = get_data("supplier")

var1 = date(1994, 1, 1)
var2 = date(1995, 1, 1)
var3 = "CANADA"
var4 = "forest"

q1 = (
    lineitem.filter(pl.col("l_shipdate").is_between(var1, var2, closed="left"))
    .group_by("l_partkey", "l_suppkey")
    .agg((pl.col("l_quantity").sum() * 0.5).alias("sum_quantity"))
)
q2 = nation.filter(pl.col("n_name") == var3)
q3 = supplier.join(q2, left_on="s_nationkey", right_on="n_nationkey")

frame = (
    part.filter(pl.col("p_name").str.starts_with(var4))
    .select(pl.col("p_partkey").unique())
    .join(partsupp, left_on="p_partkey", right_on="ps_partkey")
    .join(
        q1,
        left_on=["ps_suppkey", "p_partkey"],
        right_on=["l_suppkey", "l_partkey"],
    )
    .filter(pl.col("ps_availqty") > pl.col("sum_quantity"))
    .select(pl.col("ps_suppkey").unique())
    .join(q3, left_on="ps_suppkey", right_on="s_suppkey")
    .select("s_name", "s_address")
    .sort("s_name")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 21
lineitem = get_data("lineitem")
nation = get_data("nation")
orders = get_data("orders")
supplier = get_data("supplier")

var1 = "SAUDI ARABIA"

q1 = (
    lineitem.group_by("l_orderkey")
    .agg(pl.col("l_suppkey").len().alias("n_supp_by_order"))
    .filter(pl.col("n_supp_by_order") > 1)
    .join(
        lineitem.filter(pl.col("l_receiptdate") > pl.col("l_commitdate")),
        on="l_orderkey",
    )
)

frame = (
    q1.group_by("l_orderkey")
    .agg(pl.col("l_suppkey").len().alias("n_supp_by_order"))
    .join(q1, on="l_orderkey")
    .join(supplier, left_on="l_suppkey", right_on="s_suppkey")
    .join(nation, left_on="s_nationkey", right_on="n_nationkey")
    .join(orders, left_on="l_orderkey", right_on="o_orderkey")
    .filter(pl.col("n_supp_by_order") == 1)
    .filter(pl.col("n_name") == var1)
    .filter(pl.col("o_orderstatus") == "F")
    .group_by("s_name")
    .agg(pl.len().alias("numwait"))
    .sort(by=["numwait", "s_name"], descending=[True, False])
    .head(100)
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Query 22
customer = get_data("customer")
orders = get_data("orders")

q1 = (
    customer.with_columns(pl.col("c_phone").str.slice(0, 2).alias("cntrycode"))
    .filter(pl.col("cntrycode").str.contains("13|31|23|29|30|18|17"))
    .select("c_acctbal", "c_custkey", "cntrycode")
)

q2 = q1.filter(pl.col("c_acctbal") > 0.0).select(
    pl.col("c_acctbal").mean().alias("avg_acctbal")
)

q3 = orders.select(pl.col("o_custkey").unique()).with_columns(
    pl.col("o_custkey").alias("c_custkey")
)

frame = (
    q1.join(q3, on="c_custkey", how="left")
    .filter(pl.col("o_custkey").is_null())
    .join(q2, how="cross")
    .filter(pl.col("c_acctbal") > pl.col("avg_acctbal"))
    .group_by("cntrycode")
    .agg(
        pl.col("c_acctbal").count().alias("numcust"),
        pl.col("c_acctbal").sum().round(2).alias("totacctbal"),
    )
    .sort("cntrycode")
)

%time cpu = frame.collect(engine="streaming")
%time gpu = frame.collect(engine=engine)
gpu


In [ ]:
# Done: stop the GPU workers
engine.shutdown()
